The library `recursive-set` provides an implementation of mathematical sets that can be nested.  It provides the class `RecursiveSet` as well the the type `RecursiveSet`. 

In [ ]:
import { RecursiveSet, Value } from 'recursive-set';

In [ ]:
type RS<T extends Value> = RecursiveSet<T>;

In [ ]:
function set<T extends Value>(...elements: T[]): RS<T> {
    return new RecursiveSet(...elements);
}

# The Usual Suspects
---
There has been a burglary at a jewelry store.  The [usual suspects](https://en.wikipedia.org/wiki/The_Usual_Suspects) have been arrested.  These are
 * Aaron,
 * Bernard, and
 * Caine.
 
Furthermore, the following facts have been established:
 1. It is known that at least one of these suspects is indeed guilty.
 2. If Aaron is guilty, he has exactly one accomplice.
 3. If Bernard is innocent, then Caine is inncocent, too.
 4. If exactly two of the suspects are guilty, then Caine is one of them.
 5. If Caine is innocent, then Aaron is guilty.

It is our task to identify those suspects that are guilty.

Our first task is to define the set of propositional variables:
$$ \mathcal{P} := \{ \texttt{a}, \texttt{b}, \texttt{c} \} $$
The interpretation is that 
 * $\texttt{a}$ is true iff Aaron is guilty,
 * $\texttt{b}$ is true iff Bernard is guilty, and
 * $\texttt{c}$ is true iff Caine is guilty.  

In [ ]:
const p = set('a', 'b', 'c');
p

Our next task is to translate the facts given above into formulas from propositional logic. 

The statement "*It is known that at least one of these suspects is indeed guilty.*" is translated as follows:
$$ 
\texttt{a} \vee \texttt{b} \vee \texttt{c}. 
$$ 

In [ ]:
const f1 = 'a ∨ b ∨ c';

The statement "*If Aaron is guilty, he has exactly one accomplice.*" is harder to translate into propositional logic. The idea is to split this statement into two statements:
* If Aaron is guilty, he has at least one accomplice.</li>
* If Aaron is guilty, he has at most  one accomplice.</li>

These statements can now be translated into the following formulas:

In [ ]:
const f2 = 'a → b ∨ c';

In [ ]:
const f3 = 'a → ¬(b ∧ c)';

The statement "*If Bernard is innocent, then Caine is inncocent, too.*" is a straightforward implication:

In [ ]:
const f4 = '¬b → ¬c';

The statement "*If exactly two of the suspects are guilty, then Caine is one of them.*" is best translated into propositional logic by asking how this statement could be made false.
The statement is false if and only if the following holds:
<center>
<em>Two suspects are guilty, but Caine is innocent.</em>
 </center>
But this is only possible if and only if 
<ul>
  <li> <em>Caine is innocent</em>, </li>
  <li> <em>Aaron is guilty</em>, and  </li>
  <li> <em>Bernard is guilty</em>.  </li>
</ul>  
Hence we can translate the original statement by negating the conjunction of the statements given above and have:

In [ ]:
const f5 = '¬(¬c ∧ a ∧ b)';

The statement "*If Caine is innocent, then Aaron is guilty.*" is an implication:

In [ ]:
const f6 = '¬c → a';

We define the set `fs` of all formulas:

In [ ]:
const fs = set(f1, f2, f3, f4, f5, f6);
fs

We need to transform the strings <tt>f1</tt> to <tt>f6</tt> into nested tuples representing formulas.  To this end we import our parser for propositional formulas.

In [ ]:
import { LogicParser, Variable, Formula } from './PropositionalLogicParser'

In [ ]:
function parse(s: string): Formula {
    const parser = new LogicParser(s);
    return parser.parse();
}

Next, we transform all formulas into nested tuples:

In [ ]:
const gs = fs.map(f => parse(f));
gs

We are looking for a variable assignment $\mathcal{I}$ that satisfies all formulas in the set <tt>Fs</tt>.  As variable assignments are represented as subsets of the set $\mathcal{P}$ of propositional variables, we can just iterate over all subsets of $\mathcal{P}$.

The function $\texttt{evaluate}(F, I)$ takes a propositional formula $F$ and a propositional variable assignment $I$ and evaluates $F$ using the assignment $I$.  We have discussed the details of this function previously.

In [ ]:
function evaluate(F: Formula, I: RS<string>): boolean {
    if (typeof F == 'string') { // F is a propositional variable
        return I.has(F);
    }
    switch (F.get(0)) {
        case '⊤': 
            return true;
        case '⊥': 
            return false;
        case '¬': 
            return !evaluate(F.get(1), I);
        case '∧': 
            return evaluate(F.get(1), I) && evaluate(F.get(2), I);
        case '∨': 
            return evaluate(F.get(1), I) || evaluate(F.get(2), I);
        case '→': 
            return evaluate(F.get(1), I) <= evaluate(F.get(2), I);
        case '↔': 
            return evaluate(F.get(1), I) == evaluate(F.get(2), I); 
    }
}

The function `allTrue(Fs, I)` takes a list of propositional formula `Fs`
and a propositional variable assignment `I`.  It returns `true` only if all formulas from `Fs` are 
`true` given the variable assignment `I`.

In [ ]:
function allTrue(gs: RS<Formula>, I: RS<Variable>): boolean {
    return gs.every(f => evaluate(f, I));
}

The library `recursive-set` provides the function `powerset`, which returns the *power set* of a given set, i.e. the set of all subsets of the given set.

In [ ]:
p.powerset()

Next, we compute the set of all variable assignments that render all formulas true:

In [ ]:
const validAssignments = p.powerset().filter(i => allTrue(gs, i));
validAssignments

It turns our that there is just one propositional variable assignment that satisfies all formulas from the set <tt>Fs</tt>.  Therefore, the problem has a unique solution: Bernard and Caine are guilty, while Aaron is innocent.